In [ ]:
!pip install -q transformers datasets accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import confusion_matrix

In [ ]:
dataset = load_dataset("mteb/stsbenchmark-sts")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl.gz:   0%|          | 0.00/278k [00:00<?, ?B/s]

validation.jsonl.gz:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

test.jsonl.gz:   0%|          | 0.00/63.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [ ]:
def tokenize_function(data):
    return tokenizer(
        data["sentence1"],
        data["sentence2"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
model = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model)

bert = AutoModelForSequenceClassification.from_pretrained(
    model,
    num_labels=1,
    problem_type="regression"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
dataset_to_bert = dataset.map(tokenize_function, batched=True)

dataset_to_bert = dataset_to_bert.remove_columns(
    ["sentence1", "sentence2", "split", "genre", "dataset", "year", "sid"]
)

dataset_to_bert = dataset_to_bert.rename_column("score", "labels")

dataset_to_bert.set_format("torch")

Map:   0%|          | 0/5749 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-stsb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

In [ ]:
trainer_bert = Trainer(
    model=bert,
    args=training_args,
    train_dataset=dataset_to_bert["train"],
    eval_dataset=dataset_to_bert["validation"],
)

In [ ]:
trainer_bert.train()


Step,Training Loss
500,0.963708
1000,0.342374


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1080, training_loss=0.6242879320074011, metrics={'train_runtime': 497.4117, 'train_samples_per_second': 34.673, 'train_steps_per_second': 2.171, 'total_flos': 1134458907008256.0, 'train_loss': 0.6242879320074011, 'epoch': 3.0})

In [ ]:
predictions_bert = trainer_bert.predict(dataset_to_bert["validation"])
y_pred_bert = predictions_bert.predictions.squeeze()
y_true_bert = predictions_bert.label_ids

In [ ]:
pearson_bert = pearsonr(y_true_bert, y_pred_bert)[0]
spearman_bert = spearmanr(y_true_bert, y_pred_bert)[0]

print("BERT Pearson:", pearson_bert)
print("BERT Spearman:", spearman_bert)


BERT Pearson: 0.8868969
BERT Spearman: 0.8837705130284804


In [ ]:
y_true_bert.min(), y_true_bert.max()

(np.float32(0.0), np.float32(5.0))

In [ ]:
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model_roberta = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    problem_type="regression"
)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_function_roberta(data):
    return tokenizer(
        data["sentence1"],
        data["sentence2"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


In [ ]:
encoded_roberta = dataset.map(tokenize_function_roberta, batched=True)

encoded_roberta = encoded_roberta.remove_columns(
    ["sentence1", "sentence2", "split", "genre", "dataset", "year", "sid"]
)

encoded_roberta = encoded_roberta.rename_column("score", "labels")

encoded_roberta.set_format("torch")


Map:   0%|          | 0/5749 [00:00<?, ? examples/s]

In [ ]:
training_args_roberta = TrainingArguments(
    output_dir="./roberta-stsb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

In [ ]:
trainer_roberta = Trainer(
    model=model_roberta,
    args=training_args_roberta,
    train_dataset=encoded_roberta["train"],
    eval_dataset=encoded_roberta["validation"]
)

In [ ]:
trainer_roberta.train()

Epoch,Training Loss,Validation Loss
1,No log,0.743029
2,0.941867,0.519323
3,0.320609,0.451283


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1080, training_loss=0.603922125145241, metrics={'train_runtime': 492.0169, 'train_samples_per_second': 35.054, 'train_steps_per_second': 2.195, 'total_flos': 1134458907008256.0, 'train_loss': 0.603922125145241, 'epoch': 3.0})

In [ ]:
predictions_roberta = trainer_roberta.predict(encoded_roberta["validation"])

y_pred_roberta = predictions_roberta.predictions.squeeze()
y_true_roberta = predictions_roberta.label_ids

In [ ]:
from scipy.stats import pearsonr, spearmanr

pearson_roberta = pearsonr(y_true_roberta, y_pred_roberta)[0]
spearman_roberta = spearmanr(y_true_roberta, y_pred_roberta)[0]

print("RoBERTa Pearson:", pearson_roberta)
print("RoBERTa Spearman:", spearman_roberta)


RoBERTa Pearson: 0.90588254
RoBERTa Spearman: 0.9027413179185193
